In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model
)


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
target_month_session_names = [
    'mouse6_021322_natural_image_001',  # 仅第1个月 (Session 1)
]

recording_list = []
for file in target_month_session_names:
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{file}')
    recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
    recording_list.append(recording_recorded)

probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = si.concatenate_recordings(recording_list)
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

print(recording_f)
recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])
print(recording_cmr)

# 设置输出文件夹（与recordings_30channels_12_month.ipynb中的output_folder一致）
output_folder = '/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full'
combined_output_base = output_folder


sampling_frequency = recording_cmr.get_sampling_frequency()

segment_sample_ranges = {}  # {segment_idx: (start_sample, end_sample)}
segment_num_samples_dict = {}  # {segment_idx: num_samples}
session_names = []  # 存储每个session的名称

for i, file in enumerate(target_month_session_names):
    # 去掉文件扩展名，作为session名称
    session_name = Path(file).stem
    session_names.append(session_name)

current_sample = 0
n_segments = len(recording_list)  # session数量等于recording_list的长度

for seg_idx in range(n_segments):
    segment_num_samples = recording_list[seg_idx].get_num_samples()
    start_sample = current_sample
    end_sample = current_sample + segment_num_samples
    
    segment_sample_ranges[seg_idx] = (start_sample, end_sample)
    segment_num_samples_dict[seg_idx] = segment_num_samples
    
    session_name = session_names[seg_idx] if seg_idx < len(session_names) else f"session_{seg_idx}"
    print(f"Session {seg_idx} ({session_name}): 采样点范围 = [{start_sample}, {end_sample}), 采样点数 = {segment_num_samples}")
    
    current_sample = end_sample

print(f"\n共 {n_segments} 个sessions")



BandpassFilterRecording: 30 channels - 10000.0Hz - 1 segments - 40,000,100 samples 
                         4,000.01s (1.11 hours) - int16 dtype - 2.24 GiB
ChannelSliceRecording: 30 channels - 10000.0Hz - 1 segments - 40,000,100 samples 
                       4,000.01s (1.11 hours) - int16 dtype - 2.24 GiB
Session 0 (mouse6_021322_natural_image_001): 采样点范围 = [0, 40000100), 采样点数 = 40000100

共 1 个sessions


In [13]:
# Clique级别训练流程（适配30通道和session-based架构）
# 使用前11个月的数据进行统一训练
# 以第一个月的neuron_inf作为基准，后续月份根据这个进行筛选，去除新出现的neuron

# 创建单个包含所有30个通道的clique
probe = recording_cmr.get_probe()
probe_df = probe.to_dataframe()
all_channel_ids = probe_df['contact_ids'].astype(str).tolist()
cliques = [
    CliqueInfo(
        clique_id=0,
        device_channel_indices=list(range(len(all_channel_ids))),
        contact_ids=all_channel_ids,
        center=(probe_df['x'].mean(), probe_df['y'].mean())
    )
]

# 指定要训练的月份范围（仅第1个月，索引0）
train_session_indices = [0]
print(f"将仅使用第1个月的数据进行训练: {train_session_indices}")

# 对每个clique进行训练
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"Processing Clique {clique_id}")
    print(f"{'='*60}")
    
    # ===== 步骤1: 读取第一个月的neuron_inf作为基准 =====
    first_session_idx = 0
    first_session_name = session_names[first_session_idx]
    print(f"\n步骤1: 读取第一个月的neuron_inf作为基准")
    print(f"  Session: {first_session_name} (index: {first_session_idx})")
    
    first_session_data_folder = f'{combined_output_base}/clique_{clique_id}/{first_session_name}'
    first_neuron_inf_path = f'{first_session_data_folder}/neuron_inf.pickle'
    
    if not os.path.exists(first_neuron_inf_path):
        raise ValueError(f"第一个月的neuron_inf文件不存在: {first_neuron_inf_path}")
    
    with open(first_neuron_inf_path, 'rb') as f:
        first_neuron_inf_dict = pickle.load(f)
    
    first_neuron_inf = neuron_inf_dict_to_dataframe(first_neuron_inf_dict)
    # 获取第一个月的neuron ID列表（作为基准）
    baseline_neuron_ids = set(first_neuron_inf['Neuron'].unique())
    print(f"  基准neuron数量: {len(baseline_neuron_ids)}")
    print(f"  基准neuron IDs: {sorted(baseline_neuron_ids)}")
    
    # ===== 步骤2: 分别对每个月份进行prepare_training_data =====
    print(f"\n步骤2: 分别对每个月份进行prepare_training_data")
    
    # 创建统一的训练数据保存目录
    clique_save_dir = f'{combined_output_base}/clique_{clique_id}/train_results'
    os.makedirs(clique_save_dir, exist_ok=True)
    final_train_data_dir = Path(clique_save_dir) / "train_data"
    final_train_data_dir.mkdir(parents=True, exist_ok=True)
    
    # 存储每个月份的训练数据目录
    session_train_data_dirs = []
    sampling_rate = recording_cmr.get_sampling_frequency()
    
    for session_idx in train_session_indices:
        session_name = session_names[session_idx]
        print(f"\n  处理 Session {session_idx} ({session_name})...")
        
        session_data_folder = f'{combined_output_base}/clique_{clique_id}/{session_name}'
        neuron_inf_path = f'{session_data_folder}/neuron_inf.pickle'
        gt_detect_array_path = f'{session_data_folder}/gt_detect_array.csv'
        
        if not os.path.exists(neuron_inf_path) or not os.path.exists(gt_detect_array_path):
            print(f"    警告: {session_data_folder} 下没有找到数据文件，跳过")
            continue
        
        # 加载数据
        with open(neuron_inf_path, 'rb') as f:
            neuron_inf_dict = pickle.load(f)
        gt_detect_array = pd.read_csv(gt_detect_array_path)
        
        # 转换为DataFrame
        neuron_inf_session = neuron_inf_dict_to_dataframe(neuron_inf_dict)
        
        print(f"    原始 Neurons: {len(neuron_inf_session)}")
        print(f"    原始 Spikes: {len(gt_detect_array)}")
        
        # 筛选：只保留基准neuron（第一个月存在的neuron）
        neuron_inf_session_filtered = neuron_inf_session[neuron_inf_session['Neuron'].isin(baseline_neuron_ids)].copy()
        gt_detect_array_filtered = gt_detect_array[gt_detect_array['unit_id'].isin(baseline_neuron_ids)].copy()
        
        print(f"    筛选后 Neurons: {len(neuron_inf_session_filtered)}")
        print(f"    筛选后 Spikes: {len(gt_detect_array_filtered)}")
        
        if len(neuron_inf_session_filtered) == 0:
            print(f"    警告: 筛选后没有neuron，跳过该session")
            continue
        
        if len(gt_detect_array_filtered) == 0:
            print(f"    警告: 筛选后没有spikes，跳过该session")
            continue
        
        # 从recording_cmr中提取该session的recording（限制为前1000秒）
        start_sample, end_sample = segment_sample_ranges[session_idx]
        max_samples_per_session = int(300 * sampling_rate)  # 1000秒对应的采样点数
        
        # 限制recording为前1000秒
        session_end_sample = min(start_sample + max_samples_per_session, end_sample)
        session_recording = recording_cmr.frame_slice(start_frame=start_sample, end_frame=session_end_sample)
        recording_clique = get_recording_clique(session_recording, clique)
        
        # 筛选gt_detect_array：只保留前1000秒内的spikes
        # gt_detect_array['time']是采样点索引（相对于session开始）
        gt_detect_array_time_filtered = gt_detect_array_filtered[gt_detect_array_filtered['time'] < max_samples_per_session].copy()
        
        print(f"    限制为1000秒后 Recording: {recording_clique.get_num_samples()} samples ({recording_clique.get_num_samples() / sampling_rate:.2f} seconds)")
        print(f"    限制为1000秒后 Spikes: {len(gt_detect_array_time_filtered)}")
        
        if len(gt_detect_array_time_filtered) == 0:
            print(f"    警告: 限制为1000秒后没有spikes，跳过该session")
            continue
        
        # 为每个session创建临时保存目录
        session_save_dir = Path(clique_save_dir) / f"session_{session_idx}_{session_name}"
        session_save_dir.mkdir(parents=True, exist_ok=True)
        
        # 对该session单独调用prepare_training_data
        print(f"    对该session进行prepare_training_data...")
        session_train_data_dir = prepare_training_data(
            recording_f=recording_clique,
            gt_detect_array=gt_detect_array_time_filtered,
            neuron_inf=neuron_inf_session_filtered,
            save_dir=str(session_save_dir),
            duration_seconds=300,  # 使用1000秒
            thr_min=1,
            thr_max=10,
            distance=3,
            wlen=5,
            prominence=15,
            left_sample=10,
            right_sample=20,
            max_firing_channel=None
        )
        
        session_train_data_dirs.append(session_train_data_dir)
        print(f"    Session {session_idx} 的prepare_training_data完成")
    
    if len(session_train_data_dirs) == 0:
        print(f"  警告: 没有可用的session数据，跳过clique {clique_id}")
        continue
    
    # ===== 步骤3: 合并所有月份的训练数据 =====
    print(f"\n步骤3: 合并所有月份的训练数据")
    print(f"  合并 {len(session_train_data_dirs)} 个sessions的训练数据")
    
    # 读取所有session的训练数据
    all_X_waveform = []
    all_Y_spike_id = []
    all_Y_spike_id_noise = []
    all_X_spiketrain_time = []
    
    # 建立统一的neuron ID映射（使用第一个月的neuron作为基准）
    baseline_neuron_list = sorted(baseline_neuron_ids)
    unified_neuron_to_id = {neuron: idx for idx, neuron in enumerate(baseline_neuron_list)}
    unified_neuron_to_id[None] = -1
    unified_id_to_neuron = {idx: neuron for neuron, idx in unified_neuron_to_id.items() if neuron is not None}
    
    for session_train_data_dir in session_train_data_dirs:
        session_train_data_path = Path(session_train_data_dir)
        
        # 读取该session的训练数据
        with open(session_train_data_path / "X_waveform.pkl", "rb") as f:
            X_waveform_session = pickle.load(f)
        with open(session_train_data_path / "Y_spike_id.pkl", "rb") as f:
            Y_spike_id_session = pickle.load(f)
        with open(session_train_data_path / "Y_spike_id_noise.pkl", "rb") as f:
            Y_spike_id_noise_session = pickle.load(f)
        with open(session_train_data_path / "X_spiketrain_time.pkl", "rb") as f:
            X_spiketrain_time_session = pickle.load(f)
        with open(session_train_data_path / "neuron_mapping.pkl", "rb") as f:
            neuron_mapping_session = pickle.load(f)
        
        # 转换Y_spike_id：从session的neuron ID映射到统一的neuron ID映射
        session_id_to_neuron = neuron_mapping_session['id_to_neuron']
        Y_spike_id_unified = np.full_like(Y_spike_id_session, -1)
        
        for session_id, neuron in session_id_to_neuron.items():
            if neuron in unified_neuron_to_id:
                mask = Y_spike_id_session == session_id
                Y_spike_id_unified[mask] = unified_neuron_to_id[neuron]
        
        # 累积数据
        all_X_waveform.append(X_waveform_session)
        all_Y_spike_id.append(Y_spike_id_unified)
        all_Y_spike_id_noise.append(Y_spike_id_noise_session)
        all_X_spiketrain_time.append(X_spiketrain_time_session)
    
    # 合并所有数据
    combined_X_waveform = np.concatenate(all_X_waveform, axis=0)
    combined_Y_spike_id = np.concatenate(all_Y_spike_id, axis=0)
    combined_Y_spike_id_noise = np.concatenate(all_Y_spike_id_noise, axis=0)
    combined_X_spiketrain_time = np.concatenate(all_X_spiketrain_time, axis=0)
    
    # 保存合并后的训练数据
    unified_neuron_mapping = {
        'neuron_to_id': unified_neuron_to_id,
        'id_to_neuron': unified_id_to_neuron,
        'unique_neurons': baseline_neuron_list
    }
    
    with open(final_train_data_dir / "neuron_mapping.pkl", "wb") as f:
        pickle.dump(unified_neuron_mapping, f)
    
    with open(final_train_data_dir / "X_waveform.pkl", "wb") as f:
        pickle.dump(combined_X_waveform, f)
    
    with open(final_train_data_dir / "Y_spike_id.pkl", "wb") as f:
        pickle.dump(combined_Y_spike_id, f)
    
    with open(final_train_data_dir / "Y_spike_id_noise.pkl", "wb") as f:
        pickle.dump(combined_Y_spike_id_noise, f)
    
    with open(final_train_data_dir / "X_spiketrain_time.pkl", "wb") as f:
        pickle.dump(combined_X_spiketrain_time, f)
    
    print(f"  合并后的训练数据已保存到: {final_train_data_dir}")
    
    train_data_dir = str(final_train_data_dir)
    
    # ===== 步骤4: 训练模型（重复5次） =====
    print(f"\n步骤4: 训练模型")
    # 从第一个session获取n_channels（所有session的通道数应该相同）
    n_channels = 30  # 30通道
    n_repeats = 1
    
    for repeat_idx in range(1, n_repeats + 1):
        print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
        model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
        
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=model_save_dir,
            n_channels=n_channels,
            left_sample=10,
            right_sample=20,
            epochs=20,
            batch_size=512,
            device=None,
            early_stopping=True,
            patience=5,
            min_delta=0.0,
            use_focal_loss=True,
            focal_gamma=2.0
        )
        
        print(f"  重复训练 {repeat_idx}/{n_repeats} 完成!")
    
    print(f"\n  Clique {clique_id} 所有重复训练完成!")

print("\n所有训练完成！")


将仅使用第1个月的数据进行训练: [0]

Processing Clique 0

步骤1: 读取第一个月的neuron_inf作为基准
  Session: mouse6_021322_natural_image_001 (index: 0)


ValueError: 第一个月的neuron_inf文件不存在: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/clique_0/mouse6_021322_natural_image_001/neuron_inf.pickle